In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
================================================================================
📥 ЗАГРУЗКА КЛЮЧЕВОЙ СТАВКИ ЦБ РФ В POSTGRESQL
================================================================================

ИСТОЧНИК: Центральный банк Российской Федерации (www.cbr.ru)
ПРИЕМНИК: Таблица public.key_rate в PostgreSQL
СЕРВЕР: 109.196.102.91
БАЗА ДАННЫХ: default_db

ОПИСАНИЕ ТАБЛИЦЫ key_rate:
--------------------------------------------------------------------------------
Поле             | Тип данных | Описание
--------------------------------------------------------------------------------
date             | DATE       | Дата изменения ключевой ставки (PRIMARY KEY)
rate             | NUMERIC(10,2) | Ключевая ставка ЦБ РФ в процентах
--------------------------------------------------------------------------------
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import psycopg2
from datetime import datetime
import re
import logging
from tabulate import tabulate
import warnings
warnings.filterwarnings("ignore", category=requests.packages.urllib3.exceptions.InsecureRequestWarning)

# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Конфигурация
class Config:
    DB_HOST = "109.196.102.91"
    DB_NAME = "default_db"
    DB_USER = "maksv"
    DB_PASSWORD = r"p27Oqwp4Y,X^O^"
    DB_SCHEMA = "public"
    DB_TABLE = "key_rate"


def get_db_connection():
    """
    Создание подключения к базе данных
    """
    try:
        conn = psycopg2.connect(
            host=Config.DB_HOST,
            database=Config.DB_NAME,
            user=Config.DB_USER,
            password=Config.DB_PASSWORD
        )
        return conn
    except Exception as e:
        logger.error(f"❌ Ошибка подключения к БД: {e}")
        raise


def check_and_create_table():
    """
    Проверка существования таблицы и создание при необходимости
    """
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        # Проверяем существование таблицы
        cur.execute("""
            SELECT EXISTS (
                SELECT FROM information_schema.tables 
                WHERE table_schema = %s AND table_name = %s
            )
        """, (Config.DB_SCHEMA, Config.DB_TABLE))
        
        exists = cur.fetchone()[0]
        
        if not exists:
            logger.info(f"📋 Таблица {Config.DB_TABLE} не существует. Создаем...")
            
            # Создаем таблицу с PRIMARY KEY
            cur.execute(f"""
                CREATE TABLE {Config.DB_SCHEMA}.{Config.DB_TABLE} (
                    date DATE PRIMARY KEY,
                    rate NUMERIC(10,2)
                )
            """)
            
            conn.commit()
            logger.info(f"✅ Таблица {Config.DB_TABLE} успешно создана с PRIMARY KEY")
            return True
        else:
            # Проверяем, есть ли PRIMARY KEY на поле date
            cur.execute("""
                SELECT EXISTS (
                    SELECT 1
                    FROM information_schema.table_constraints tc
                    JOIN information_schema.constraint_column_usage ccu 
                        ON tc.constraint_name = ccu.constraint_name
                    WHERE tc.constraint_type = 'PRIMARY KEY'
                        AND tc.table_schema = %s
                        AND tc.table_name = %s
                        AND ccu.column_name = 'date'
                )
            """, (Config.DB_SCHEMA, Config.DB_TABLE))
            
            has_primary_key = cur.fetchone()[0]
            
            if not has_primary_key:
                logger.warning(f"⚠️ Таблица существует, но нет PRIMARY KEY на поле date")
                logger.info(f"🔄 Добавляем PRIMARY KEY...")
                
                # Добавляем PRIMARY KEY
                cur.execute(f"""
                    ALTER TABLE {Config.DB_SCHEMA}.{Config.DB_TABLE}
                    ADD CONSTRAINT key_rate_pkey PRIMARY KEY (date)
                """)
                
                conn.commit()
                logger.info(f"✅ PRIMARY KEY успешно добавлен")
            
            logger.info(f"✅ Таблица {Config.DB_TABLE} существует и готова к работе")
            return True
        
    except Exception as e:
        logger.error(f"❌ Ошибка при проверке/создании таблицы: {e}")
        if conn:
            conn.rollback()
        return False
    finally:
        if conn:
            conn.close()


def get_existing_dates_from_db():
    """
    Получение всех существующих дат из базы данных
    """
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        cur.execute(f"""
            SELECT date 
            FROM {Config.DB_SCHEMA}.{Config.DB_TABLE}
        """)
        
        existing_dates = {row[0] for row in cur.fetchall()}
        logger.info(f"📋 В БД найдено {len(existing_dates)} существующих записей")
        
        return existing_dates
        
    except Exception as e:
        logger.error(f"❌ Ошибка при получении существующих дат из БД: {e}")
        return set()
    finally:
        if conn:
            conn.close()


def get_latest_date_from_db():
    """
    Получение последней даты из базы данных
    """
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        cur.execute(f"""
            SELECT MAX(date) 
            FROM {Config.DB_SCHEMA}.{Config.DB_TABLE}
        """)
        
        result = cur.fetchone()
        return result[0] if result and result[0] else None
        
    except Exception as e:
        logger.error(f"❌ Ошибка при получении последней даты из БД: {e}")
        return None
    finally:
        if conn:
            conn.close()


def parse_cbr_key_rate():
    """
    Парсинг всех исторических данных ключевой ставки ЦБ РФ с официального сайта
    """
    url = "https://www.cbr.ru/hd_base/KeyRate/"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        logger.info(f"🌐 Запрос к ЦБ РФ для получения исторических данных ключевой ставки")
        
        response = requests.get(url, headers=headers, verify=False, timeout=30)
        response.encoding = 'utf-8'
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Поиск данных в JavaScript
        scripts = soup.find_all('script')
        
        data_pattern = r'var settings = ({.*?});'
        categories = None
        values = None
        
        for script in scripts:
            if script.string and 'var settings' in script.string:
                match = re.search(data_pattern, script.string, re.DOTALL)
                if match:
                    settings_str = match.group(1)
                    
                    categories_match = re.search(r'"categories":\[(.*?)\]', settings_str)
                    if categories_match:
                        categories_str = categories_match.group(1)
                        categories = [c.strip('"') for c in categories_str.split(',')]
                    
                    data_match = re.search(r'"data":\[(.*?)\](?=,\s*"color")', settings_str)
                    if data_match:
                        data_str = data_match.group(1)
                        values = [float(x) for x in data_str.split(',')]
                    
                    break
        
        if categories and values and len(categories) == len(values):
            df = pd.DataFrame({
                'date': pd.to_datetime(categories, format='%d.%m.%Y'),
                'rate': values
            })
            df = df.sort_values('date').reset_index(drop=True)
            logger.info(f"✅ Получено {len(df)} записей из JavaScript")
            return df
        
        # Если данные не найдены в JavaScript, ищем в HTML таблице
        logger.info("🔄 Данные не найдены в JSON, ищем в HTML таблице...")
        table = soup.find('table', class_='data')
        if table:
            data = []
            rows = table.find_all('tr')[1:]  # Пропускаем заголовок
            for row in rows:
                cols = row.find_all('td')
                if len(cols) == 2:
                    date_str = cols[0].text.strip()
                    rate_str = cols[1].text.strip().replace(',', '.')
                    try:
                        rate = float(rate_str)
                        data.append({'date': date_str, 'rate': rate})
                    except ValueError:
                        continue
            
            if data:
                df = pd.DataFrame(data)
                df['date'] = pd.to_datetime(df['date'], format='%d.%m.%Y')
                df = df.sort_values('date').reset_index(drop=True)
                logger.info(f"✅ Получено {len(df)} записей из HTML таблицы")
                return df
        
        logger.error("❌ Не удалось найти данные на странице")
        return None
            
    except requests.RequestException as e:
        logger.error(f"❌ Ошибка сети: {e}")
        return None
    except Exception as e:
        logger.error(f"❌ Ошибка при парсинге: {e}")
        return None


def get_new_records_only(df, existing_dates):
    """
    Фильтрация только новых записей, которых нет в базе
    """
    if not existing_dates:
        logger.info("📋 База данных пуста - будут добавлены все записи")
        return df
    
    # Фильтруем записи, которых нет в existing_dates
    new_records = df[~df['date'].dt.date.isin(existing_dates)]
    
    if len(new_records) > 0:
        logger.info(f"🆕 Найдено {len(new_records)} новых записей для добавления")
        if len(new_records) < len(df):
            latest_existing = max(existing_dates) if existing_dates else None
            logger.info(f"📅 Последняя существующая дата: {latest_existing}")
            logger.info(f"📅 Новые данные до: {new_records['date'].max().date()}")
    else:
        latest_existing = max(existing_dates) if existing_dates else None
        logger.info(f"✅ Данные актуальны. Последняя запись в БД: {latest_existing}")
    
    return new_records


def get_db_statistics():
    """
    Получение статистики из базы данных
    """
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        cur.execute("""
            SELECT 
                COUNT(*) as total_records,
                MIN(date) as first_date,
                MAX(date) as last_date,
                MIN(rate) as min_rate,
                MAX(rate) as max_rate,
                ROUND(AVG(rate)::numeric, 2) as avg_rate
            FROM public.key_rate
        """)
        
        stats = cur.fetchone()
        
        cur.execute("""
            SELECT date, rate 
            FROM public.key_rate 
            ORDER BY date DESC 
            LIMIT 1
        """)
        
        last = cur.fetchone()
        
        return {
            'total_records': stats[0] or 0,
            'first_date': stats[1],
            'last_date': stats[2],
            'min_rate': stats[3],
            'max_rate': stats[4],
            'avg_rate': stats[5],
            'current_date': last[0] if last else None,
            'current_rate': last[1] if last else None
        }
        
    except Exception as e:
        logger.error(f"❌ Ошибка при получении статистики: {e}")
        return None
    finally:
        if conn:
            conn.close()


def display_statistics(stats, title="📊 СТАТИСТИКА КЛЮЧЕВОЙ СТАВКИ В БАЗЕ"):
    """
    Отображение статистики в табличном виде
    """
    if not stats or stats['total_records'] == 0:
        print("\n❌ В базе нет данных по ключевой ставке")
        return
    
    print("\n" + "="*80)
    print(title)
    print("="*80)
    
    stats_data = [
        ["Всего записей", f"{stats['total_records']}"],
        ["Период данных", f"{stats['first_date']} - {stats['last_date']}"],
        ["Минимальная ставка", f"{stats['min_rate']:.2f}%"],
        ["Максимальная ставка", f"{stats['max_rate']:.2f}%"],
        ["Средняя ставка", f"{stats['avg_rate']:.2f}%"],
        ["Текущая ставка", f"{stats['current_rate']:.2f}% (на {stats['current_date']})"]
    ]
    
    print(tabulate(stats_data, headers=["Показатель", "Значение"], tablefmt="grid"))
    print("="*80)


def save_to_postgresql(df):
    """
    Сохранение только новых данных в PostgreSQL
    Использует INSERT без ON CONFLICT, так как может не быть уникального约束
    """
    if df is None or df.empty:
        logger.warning("⚠️ Нет данных для сохранения")
        return 0
    
    conn = None
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        
        records_saved = 0
        records_skipped = 0
        
        # Получаем существующие даты перед вставкой
        cur.execute(f"SELECT date FROM {Config.DB_SCHEMA}.{Config.DB_TABLE}")
        existing_dates = {row[0] for row in cur.fetchall()}
        
        for _, row in df.iterrows():
            date_obj = row['date'].date()
            rate_value = round(float(row['rate']), 2)
            
            # Проверяем, есть ли уже такая дата
            if date_obj in existing_dates:
                records_skipped += 1
                continue
            
            # Вставляем новую запись
            insert_sql = f"""
                INSERT INTO {Config.DB_SCHEMA}.{Config.DB_TABLE} 
                (date, rate)
                VALUES (%s, %s)
            """
            
            cur.execute(insert_sql, (date_obj, rate_value))
            records_saved += 1
            existing_dates.add(date_obj)  # Добавляем в множество для последующих проверок
        
        conn.commit()
        
        if records_saved > 0:
            logger.info(f"✅ Добавлено {records_saved} новых записей")
        if records_skipped > 0:
            logger.info(f"⏭️ Пропущено {records_skipped} существующих записей")
        
        if records_saved == 0 and records_skipped == 0:
            logger.info("✅ Новых записей для добавления не найдено")
        
        return records_saved
        
    except Exception as e:
        logger.error(f"❌ Ошибка при сохранении в БД: {e}")
        if conn:
            conn.rollback()
        return 0
    finally:
        if conn:
            conn.close()


def update_key_rate():
    """
    Основная функция обновления ключевой ставки
    Добавляет только новые записи, которых нет в базе
    """
    print("\n" + "="*100)
    print("💰 ОБНОВЛЕНИЕ КЛЮЧЕВОЙ СТАВКИ ЦБ РФ В POSTGRESQL")
    print("="*100)
    print(f"🖥️  Сервер: {Config.DB_HOST}")
    print(f"🗄️  База данных: {Config.DB_NAME}")
    print(f"📋 Таблица: {Config.DB_SCHEMA}.{Config.DB_TABLE}")
    print("="*100)
    
    # Проверяем/создаем таблицу
    if not check_and_create_table():
        logger.error("❌ Не удалось настроить таблицу")
        return None
    
    # Получаем текущую статистику
    current_stats = get_db_statistics()
    display_statistics(current_stats, "📊 ТЕКУЩАЯ СТАТИСТИКА В БАЗЕ")
    
    # Получаем все существующие даты из базы
    existing_dates = get_existing_dates_from_db()
    
    # Загружаем все данные с сайта ЦБ
    logger.info("\n🔄 Загрузка данных с сайта ЦБ РФ...")
    df_all = parse_cbr_key_rate()
    
    if df_all is None or df_all.empty:
        logger.error("❌ Не удалось получить данные с сайта ЦБ РФ")
        return None
    
    # Фильтруем только новые записи
    df_new = get_new_records_only(df_all, existing_dates)
    
    if df_new.empty:
        print("\n" + "="*80)
        print("✅ ДАННЫЕ АКТУАЛЬНЫ")
        print("="*80)
        print(f"📅 Последняя дата в БД: {current_stats['last_date']}")
        print(f"📊 Всего записей в БД: {current_stats['total_records']}")
        print("="*80)
        return {
            'records_parsed': len(df_all),
            'records_saved': 0,
            'total_in_db': current_stats['total_records'],
            'first_date': current_stats['first_date'],
            'last_date': current_stats['last_date'],
            'current_rate': current_stats['current_rate'],
            'status': 'up_to_date'
        }
    
    # Показываем информацию о новых данных
    print("\n📊 НОВЫЕ ДАННЫЕ ДЛЯ ДОБАВЛЕНИЯ:")
    print("-" * 60)
    print(f"📅 Период: {df_new['date'].min().date()} - {df_new['date'].max().date()}")
    print(f"📊 Новых записей: {len(df_new)}")
    print(f"📉 Диапазон ставок: {df_new['rate'].min()}% - {df_new['rate'].max()}%")
    print(f"💰 Новая ставка: {df_new.iloc[-1]['rate']}% (на {df_new.iloc[-1]['date'].date()})")
    print("-" * 60)
    
    # Сохраняем новые записи
    records_saved = save_to_postgresql(df_new)
    
    # Получаем обновленную статистику
    new_stats = get_db_statistics()
    display_statistics(new_stats, "📊 ОБНОВЛЕННАЯ СТАТИСТИКА В БАЗЕ")
    
    print("\n" + "="*80)
    print("✅ ОПЕРАЦИЯ ЗАВЕРШЕНА")
    print("="*80)
    print(f"📥 Добавлено в БД: {records_saved} записей")
    print(f"📊 Всего в БД: {new_stats['total_records']} записей")
    print(f"📅 Период: {new_stats['first_date']} - {new_stats['last_date']}")
    print(f"💰 Текущая ставка: {new_stats['current_rate']:.2f}% (на {new_stats['current_date']})")
    print("="*80)
    
    return {
        'records_parsed': len(df_all),
        'records_saved': records_saved,
        'total_in_db': new_stats['total_records'],
        'first_date': new_stats['first_date'],
        'last_date': new_stats['last_date'],
        'current_rate': new_stats['current_rate'],
        'status': 'updated' if records_saved > 0 else 'no_changes'
    }


def main():
    """
    Основная функция - только обновление данных при необходимости
    """
    # Запускаем обновление
    results = update_key_rate()


if __name__ == "__main__":
    main()

2026-03-01 16:33:10,049 - __main__ - WARNING - ⚠️ Таблица существует, но нет PRIMARY KEY на поле date
2026-03-01 16:33:10,057 - __main__ - INFO - 🔄 Добавляем PRIMARY KEY...
2026-03-01 16:33:10,087 - __main__ - INFO - ✅ PRIMARY KEY успешно добавлен
2026-03-01 16:33:10,088 - __main__ - INFO - ✅ Таблица key_rate существует и готова к работе



💰 ОБНОВЛЕНИЕ КЛЮЧЕВОЙ СТАВКИ ЦБ РФ В POSTGRESQL
🖥️  Сервер: 109.196.102.91
🗄️  База данных: default_db
📋 Таблица: public.key_rate


2026-03-01 16:33:10,289 - __main__ - INFO - 📋 В БД найдено 3116 существующих записей
2026-03-01 16:33:10,293 - __main__ - INFO - 
🔄 Загрузка данных с сайта ЦБ РФ...
2026-03-01 16:33:10,293 - __main__ - INFO - 🌐 Запрос к ЦБ РФ для получения исторических данных ключевой ставки



📊 ТЕКУЩАЯ СТАТИСТИКА В БАЗЕ
+---------------------+-------------------------+
| Показатель          | Значение                |
+=====================+=========================+
| Всего записей       | 3116                    |
+---------------------+-------------------------+
| Период данных       | 2013-09-17 - 2026-02-25 |
+---------------------+-------------------------+
| Минимальная ставка  | 4.25%                   |
+---------------------+-------------------------+
| Максимальная ставка | 21.00%                  |
+---------------------+-------------------------+
| Средняя ставка      | 10.21%                  |
+---------------------+-------------------------+
| Текущая ставка      | 15.50% (на 2026-02-25)  |
+---------------------+-------------------------+


2026-03-01 16:33:10,576 - __main__ - INFO - ✅ Получено 3118 записей из JavaScript
2026-03-01 16:33:10,605 - __main__ - INFO - 🆕 Найдено 2 новых записей для добавления
2026-03-01 16:33:10,605 - __main__ - INFO - 📅 Последняя существующая дата: 2026-02-25
2026-03-01 16:33:10,608 - __main__ - INFO - 📅 Новые данные до: 2026-02-27
2026-03-01 16:33:10,741 - __main__ - INFO - ✅ Добавлено 2 новых записей



📊 НОВЫЕ ДАННЫЕ ДЛЯ ДОБАВЛЕНИЯ:
------------------------------------------------------------
📅 Период: 2026-02-26 - 2026-02-27
📊 Новых записей: 2
📉 Диапазон ставок: 15.5% - 15.5%
💰 Новая ставка: 15.5% (на 2026-02-27)
------------------------------------------------------------

📊 ОБНОВЛЕННАЯ СТАТИСТИКА В БАЗЕ
+---------------------+-------------------------+
| Показатель          | Значение                |
+=====================+=========================+
| Всего записей       | 3118                    |
+---------------------+-------------------------+
| Период данных       | 2013-09-17 - 2026-02-27 |
+---------------------+-------------------------+
| Минимальная ставка  | 4.25%                   |
+---------------------+-------------------------+
| Максимальная ставка | 21.00%                  |
+---------------------+-------------------------+
| Средняя ставка      | 10.22%                  |
+---------------------+-------------------------+
| Текущая ставка      | 15.50% (на 2026